# Comparing different search methods for Wikipedia

In this notebook we study several different ways to query a Wikipedia database, including:
*   Keyword search
*   Dense retrieval
*   Reranking

Furthermore, we combine the power of search with Cohere's Chat endpoint in order to output accurate answers in sentence format to a query.

This notebook accompanies the [Semantic Search](https://docs.cohere.com/docs/intro-semantic-search) section of LLM University.

## Setup

We'll start by installing the tools we'll need and then importing them.

In [ ]:
# !pip uninstall -y weaviate-client
!pip install -U weaviate-client cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.5/660.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.84.0
    Uninstalling grpcio-1.84.0:
      Successfully uninstalled grpcio-1.84.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.84.0 requires grpcio>=1.84.0, but you have grpcio 1.78.0 which is incompatible.


In [ ]:
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery
import cohere

Fill in your Cohere API key in the next cell. To do this, begin by [signing up to Cohere](https://os.cohere.ai/) (for free!) if you haven't yet. Then get your API key [here] https://dashboard.cohere.com/api-keys).

In [ ]:
# Fill in your API key here. Remember to not share publicly
cohere_api_key = "API_KEY"

co = cohere.Client(cohere_api_key)

In [ ]:
#Connect to the Weaviate demo database containing 10M wikipedia vectors
auth_config = Auth.api_key("AUTH_API_KEY")
client = weaviate.connect_to_weaviate_cloud(
    # cluster_url="https://cohere-demo.weaviate.network/",
    cluster_url="CLUSTER_URL",
    auth_credentials=auth_config,
    headers={
      "X-Cohere-Api-Key": cohere_api_key,
    }
)

client.is_ready() # check if True

True

# Keyword Search

This section accompanies the [Keyword Search](https://docs.cohere.com/docs/keyword-search) chapter of LLM University.

We'll search for two queries using keyword search.

* Simple query: "Who discovered penicillin?" (Answer: Alexander Fleming)

* Hard query: "Who was the first person to win two Nobel prizes?" (Answer: Marie Curie)

You will notice that keyword search performs very well with the simple query, and not so well with the hard one.

In [ ]:
def keyword_search(query, results_lang='en', num_results=10):
  properties = ["text", "title", "url", "views", "lang"]

  articles = client.collections.get("Articles")

  response = articles.query.bm25(
        query=query,
        filters=Filter.by_property("lang").equal(results_lang),
        limit=num_results,
        return_properties=properties
    )

  result = []

  for obj in response.objects:
      result.append({
          **obj.properties,
          "distance": obj.metadata.distance
      })

  return result

In [ ]:
def print_result(result):
  """Print results with colorful formatting"""
  for item in result:
    print(f"\033[95m{item['title']} ({item['views']}) \033[0m")
    print(f"\033[4m{item['url']}\033[0m")
    print(item['text'])
    print('\n')

In [ ]:
collections = client.collections.list_all()
print(collections)
print(list(client.collections.list_all().keys()))

{}
[]


In [ ]:
simple_query = "Who discovered penicillin?"
keyword_search_results_simple = keyword_search(simple_query)
print_result(keyword_search_results_simple)

In [ ]:
hard_query = "Who was the first person to win two nobel prizes?"
keyword_search_results_hard = keyword_search(hard_query)
print_result(keyword_search_results_hard)

#Dense Retrieval

This section accompanies the [Dense Retrieval](https://docs.cohere.com/docs/dense-retrieval) chapter of LLM University.

Now we will use dense retrieval to search the answers for the two queries. Now you will notice that the results are good for both queries.

In [ ]:
# This function performs dense retrieval
def dense_retrieval (query, results_lang='en', num_results=10):
  """
  Query the vectors database and return the top results.
  Parameters
  ----------
    query: str
      The search query

    results_lang: str (optional)
      Retrieve results only in the specified language.
      The demo dataset has those languages:
      en, de, fr, es, it, ja, ar, zh, ko, hi
  """
  properties = ["text", "title", "url", "views", "lang"]
  nearText = {"concepts": [query]}

  articles = client.collections.get("Articles")

  response = articles.query.near_text(
        query=query,
        filters=Filter.by_property("lang").equal(results_lang),
        limit=num_results,
        return_properties=properties
        return_metadata=MetadataQuery(distance=True)
    )

  result = []

  for obj in response.objects:
      result.append({
          **obj.properties,
          "distance": obj.metadata.distance
      })


In [ ]:
simple_query = "Who discovered penicillin?"
dense_retrieval_results_simple = dense_retrieval(simple_query)
print_result(dense_retrieval_results_simple)

In [ ]:
hard_query = "Who was the first person to win two nobel prizes?"
dense_retrieval_results_hard = dense_retrieval(hard_query)
print_result(dense_retrieval_results_hard)

### Searching in other languages
Changing the `results_lang` parameter to any of the following: en, de, fr, es, it, ja, ar, zh, ko, hi (the available languages in the demo) allows you to get results in any language you want. For example, here are the results to the hard query in Arabic.

In [ ]:
arabic_results = dense_retrieval(hard_query, results_lang='ar')
print_result(arabic_results)

The query can also be in any other language. Here are the French results to a query in Spanish.

In [ ]:
spanish_query = "Quien descubrio la penicilina?"
french_results = dense_retrieval(spanish_query, results_lang='fr')
print_result(french_results)

# ReRank

This section accompanies the [Reranking](https://docs.cohere.com/docs/reranking-2) chapter of LLM University.

Rerank is a powerful method that will enhance any search model. In short, rerank takes a query and a set of responses (or documents), and will surface the ones that are the most relevant as answers to the query. We'll use Rerank to improve keyword search with the hard query.

In [ ]:
def rerank_responses (query, responses, num_responses=3):
  reranked_responses = co.rerank(
      query=query,
      documents=responses,
      top_n=num_responses,
      model='rerank-english-v3.0',
      return_documents=True
  )

  return reranked_responses

In [ ]:
hard_query = "Who was the first person to win two nobel prizes?"
keyword_searches_to_improve = keyword_search(hard_query, num_results = 100)

In [ ]:
for r in keyword_searches_to_improve[:20]:
  print(r['title'], ':', r['text'])

In [ ]:
reranked_keyword_responses = rerank_responses(hard_query, keyword_searches_to_improve, num_responses=3)

In [ ]:
for idx, r in enumerate(reranked_keyword_responses.results):
  print(f"Document Rank: {idx+1}, Document Index: {r.index}")
  print(f"Title: {r.document.title}")
  print(f"URL: {r.document.url}")
  print(f"Document: {r.document.text}")
  print(f"Relevance Score: {r.relevance_score:.2f}")
  print("\n")

# Generating responses

This section accompanies the [Generating Answers](https://docs.cohere.com/docs/generating-answers) chapter of LLM University.

Generative models are great at talking, but when it comes to answer questions with facts, they are prone to hallucinations. In other words, they can answer with the wrong answer. To prevent this, we first search for the documents that are relevant to the query (using dense retrieval, but we can use any method). We then feed them to the generative model, and instruct it to answer the question from the information from those documents.

The query is "How many people have won more than one Nobel prize?". You will notice that the model generates wrong answers, but when combined with search, it'll generate the correct answers.

In [ ]:
query = "How many people have won more than one Nobel prize?"

In [ ]:
prediction_without_search = [
  co.chat(
      message=query,
      max_tokens=50,
  ) for _ in range(5)
]

In [ ]:
for p in prediction_without_search:
  print(p.text)

In [ ]:
responses = dense_retrieval(query, num_results=20)
print_result(responses)

context = [r['text'] for r in responses]
context[:10]

In [ ]:
prompt = f"""
Use the information provided below to answer the question at the end. If the answer to the question is not contained in the provided information, say 'The answer is not in the context'.
---
Context Information:
{context}
---
Question:
{query}
"""

In [ ]:
prediction_with_search = [
    co.chat(
        message=prompt,
        max_tokens=50
    ) for _ in range(5)
]

for p in prediction_with_search:
  print(p.text)